# Word sense classification with LLMs: ICL vs LoRA

Model: [unsloth/Qwen2.5-3B-Instruct-bnb-4bit](https://huggingface.co/unsloth/Qwen2.5-3B-Instruct-bnb-4bit) (instruction-tuned, 4-bit pre-quantized).
Run on Kaggle with a T4 GPU and Internet enabled.

## Setup
Run the install cell once per new session. If Kaggle asks to restart, restart and continue from the next cell.

In [ ]:
# unsloth installs compatible versions of transformers, trl, peft, bitsandbytes, accelerate and datasets
%pip install unsloth

In [ ]:
# unsloth must be imported before transformers / trl / peft so that its patches apply
from unsloth import FastLanguageModel

import math, time
import torch, transformers, trl, peft, bitsandbytes
import pandas as pd
import matplotlib.pyplot as plt
from datasets import Dataset

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available")
print("transformers", transformers.__version__, "| trl", trl.__version__,
      "| peft", peft.__version__, "| bitsandbytes", bitsandbytes.__version__)

## Shared code and data
The next cell is shared with the ICL part: same data preparation, input format, decoding, parsing and metrics.

In [ ]:
# Shared code: identical to lab1_shared.py (used by both the ICL and LoRA parts)

import os
import re
import gzip
import random
import urllib.request

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report


# =====================================================================
# 1. Fixed settings -- any change must be made by both team members
# =====================================================================

MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
SEED = 42
MAX_NEW_TOKENS = 10        # labels are short; 10 new tokens are enough

DEV_SUBSET_N = None        # None = full rand-dev; set an int (e.g. 300) to tune on a fixed subset
DROP_LABELS = []           # hypersenses to exclude (e.g. ["other"]), if we decide to

DATA_URL = ("http://www.linguist.univ-paris-diderot.fr/~mcandito/divers/"
            "super_wiktionary_labeled_data_simplified.noex.25labels.tsv.gz")

# Supersense -> hypersense mapping, copied from the lab notebook
SUPER2HYPER = {
    'artifact': 'inanimate',
    'body': 'inanimate',
    'food': 'inanimate',
    'object': 'inanimate',
    'plant': 'inanimate',
    'substance': 'inanimate',
    'act': 'dynamic_situation',
    'event': 'dynamic_situation',
    'phenomenon': 'dynamic_situation',
    'animal': 'animate',
    'person': 'animate',
    'groupxperson': 'animate',
    'cognition': 'information',
    'communication': 'information',
    'quantity': 'quantity',
    'attribute': 'stative_situation',
    'feeling': 'stative_situation',
    'relation': 'stative_situation',
    'state': 'stative_situation',
    'time': 'time',
    'possession': 'possession',
    'institution': 'institution',
    'act*cognition': 'dynamic_situation*information',
    'artifact*cognition': 'inanimate*information',
    'other': 'other',
}

# Field names of the input, depending on the prompt language
FIELDS = {
    "en": ("Lemma", "Definition"),
    "fr": ("Lemme", "Définition"),
}


def set_seed(seed=SEED):
    """Fix Python, NumPy and PyTorch random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


# =====================================================================
# 2. Data loading and preparation
# =====================================================================

def load_data(url=DATA_URL):
    """Download the gzipped TSV (if not already present) and load it."""
    filename = url.split('/')[-1]
    if not os.path.exists(filename):
        urllib.request.urlretrieve(url, filename)
    with gzip.open(filename, 'rt', encoding='utf-8') as f:
        return pd.read_csv(f, sep='\t')


def prepare_data(df):
    """Add the hypersense column, drop protect-frsemcor, split by the 'set' column.
    Returns (splits, labels):
      splits: dict with keys 'train', 'rand-dev', 'rand-test', 'freq-test'
      labels: sorted list of hypersense labels, shared by both methods"""
    df = df.copy()
    df['hypersense'] = df['supersense'].map(SUPER2HYPER)
    missing = df.loc[df['hypersense'].isna(), 'supersense'].unique()
    assert len(missing) == 0, f"Supersenses missing from SUPER2HYPER: {missing}"

    df = df[df['set'] != 'protect-frsemcor']
    if DROP_LABELS:
        df = df[~df['hypersense'].isin(DROP_LABELS)]

    # Sets other than the four below (if any) are not used
    print(df['set'].value_counts(), "\n")

    names = ['train', 'rand-dev', 'rand-test', 'freq-test']
    splits = {n: df[df['set'] == n].reset_index(drop=True) for n in names}
    if DEV_SUBSET_N:
        splits['rand-dev'] = (splits['rand-dev']
                              .sample(DEV_SUBSET_N, random_state=SEED)
                              .reset_index(drop=True))

    labels = sorted(df['hypersense'].unique())
    for n in names:
        print(f"{n:10s}: {len(splits[n])} senses")
        assert len(splits[n]) > 0, f"Split '{n}' is empty: check the values of the 'set' column"
    print("Labels:", labels)
    return splits, labels


# =====================================================================
# 3. Input format -- used by both ICL and LoRA
# =====================================================================

def format_input(row, lang="en", use_lemma=True):
    """Turn one word sense into the input text, e.g.
         Lemma: béruguette
         Definition: Variété d'olive allongée à pulpe ferme."""
    f_lemma, f_def = FIELDS[lang]
    text = f"{f_def}: {row['definition']}"
    if use_lemma:
        text = f"{f_lemma}: {row['lemma']}\n" + text
    return text


# =====================================================================
# 4. Batched generation -- used by both methods at inference time
# =====================================================================

def generate_batch(model, tokenizer, messages_list, batch_size=16):
    """messages_list: one chat (list of messages) per instance.
    Returns the raw generated strings, in the same order as messages_list.
    Greedy decoding, MAX_NEW_TOKENS, left padding, batches sorted by length."""
    import torch

    old_side = tokenizer.padding_side
    tokenizer.padding_side = "left"              # required for batched generation
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    prompts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
               for m in messages_list]
    # Sort by length so each batch needs little padding; restore order at the end
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    outputs = [None] * len(prompts)

    for start in range(0, len(order), batch_size):
        idx = order[start:start + batch_size]
        # The chat template already contains the special tokens
        enc = tokenizer([prompts[i] for i in idx], return_tensors="pt",
                        padding=True, add_special_tokens=False).to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                 pad_token_id=tokenizer.pad_token_id)
        texts = tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:],
                                       skip_special_tokens=True)
        for i, t in zip(idx, texts):
            outputs[i] = t.strip()

    tokenizer.padding_side = old_side
    return outputs


# =====================================================================
# 5. Parsing and evaluation
# =====================================================================

def parse_output(text, labels):
    """Extract a label from the generated text. Returns (prediction, status) with
    status 'valid' (exactly one label), 'none' (no label) or 'multiple' (several labels).
    Boundaries exclude letters, digits, '_' and '*', so 'animate' does not match
    inside 'inanimate', and 'inanimate' does not match inside 'inanimate*information'."""
    text = text.lower()
    found = [l for l in labels
             if re.search(r'(?<![\w*])' + re.escape(l.lower()) + r'(?![\w*])', text)]
    if len(found) == 1:
        return found[0], "valid"
    return "INVALID", ("none" if not found else "multiple")


def evaluate(rows_df, raw_outputs, labels, verbose=True):
    """Returns (metrics, res):
      metrics: accuracy and macro-F1 (invalid outputs count as errors), invalid rates
      res: one row per instance, for saving and error analysis"""
    parsed = [parse_output(t, labels) for t in raw_outputs]
    res = pd.DataFrame({
        "sense_id":   rows_df["sense_id"].values,
        "lemma":      rows_df["lemma"].values,
        "definition": rows_df["definition"].values,
        "gold":       rows_df["hypersense"].values,
        "raw":        raw_outputs,
        "pred":       [p for p, _ in parsed],
        "status":     [s for _, s in parsed],
    })
    valid = res["status"] == "valid"
    # Macro-F1 is averaged over the labels that occur in this split only,
    # so that a label absent from the split does not pull the average down
    present = [l for l in labels if l in set(res["gold"])]
    metrics = {
        "n":                len(res),
        "accuracy":         accuracy_score(res["gold"], res["pred"]),
        "macro_f1":         f1_score(res["gold"], res["pred"], labels=present,
                                     average="macro", zero_division=0),
        "invalid_rate":     1 - valid.mean(),
        "no_label_rate":    (res["status"] == "none").mean(),
        "multi_label_rate": (res["status"] == "multiple").mean(),
        "acc_on_valid":     accuracy_score(res.loc[valid, "gold"], res.loc[valid, "pred"])
                            if valid.any() else 0.0,
    }
    if verbose:
        for k, v in metrics.items():
            print(f"{k:18s}: {v:.4f}" if isinstance(v, float) else f"{k:18s}: {v}")
        print(classification_report(res["gold"], res["pred"], labels=labels, zero_division=0))
    return metrics, res


# =====================================================================
# 6. Saving results -- {run}_{split}_preds.csv + one shared results_summary.csv
# =====================================================================

def save_results(run_name, split_name, metrics, res, out_dir, config=None):
    """run_name: e.g. 'E1', 'L1'; split_name: 'rand-dev', 'rand-test' or 'freq-test';
    config: optional dict describing the run, e.g. {'method': 'ICL', 'lang': 'en', 'lemma': True}"""
    os.makedirs(out_dir, exist_ok=True)
    res.to_csv(os.path.join(out_dir, f"{run_name}_{split_name}_preds.csv"), index=False)

    row = {"run": run_name, "split": split_name, **(config or {}), **metrics}
    summary_path = os.path.join(out_dir, "results_summary.csv")
    summary = pd.DataFrame([row])
    if os.path.exists(summary_path):
        summary = pd.concat([pd.read_csv(summary_path), summary], ignore_index=True)
    summary.to_csv(summary_path, index=False)
    print(f"Saved: {run_name} / {split_name}")

In [ ]:
set_seed()
splits, labels = prepare_data(load_data())

# LoRA fine-tuning

## Experiment settings
Only this cell changes between runs (L1: with lemma, L2: without lemma).
Run the whole notebook from the top for each run, so that every run starts from the base model.

In [ ]:
RUN_NAME   = "L1"          # L1: with lemma; L2: without lemma
USE_LEMMA  = True
LANG       = "en"          # must match the ICL setting
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",      # attention
                  "gate_proj", "up_proj", "down_proj"]         # MLP
SMOKE_TEST = True          # True: 200 training instances, 1 epoch, to check the pipeline

MAX_SEQ_LEN = 512          # upper bound on tokens per training example
OUT_DIR = "/kaggle/working"
RUN_DIR = f"{OUT_DIR}/{RUN_NAME}{'_smoke' if SMOKE_TEST else ''}"

train = splits["train"]
dev   = splits["rand-dev"]
if SMOKE_TEST:
    train = train.sample(200, random_state=SEED)
print(f"train: {len(train)} | rand-dev: {len(dev)}")

## Model with LoRA adapters

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_NAME, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True)

# Base weights stay frozen; only the low-rank matrices added to TARGET_MODULES are trained
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=TARGET_MODULES,
    bias="none",
    use_gradient_checkpointing="unsloth",   # trades compute for memory
    random_state=SEED,
)
model.print_trainable_parameters()

## Training data
Each instance becomes a chat: the user turn contains a short instruction and the input, the assistant turn contains the gold hypersense.

In [ ]:
INSTRUCTION = {
    "en": "Classify this French word sense into a hypersense.",
    "fr": "Classe ce sens de mot dans une hypersense.",
}

def make_messages(row, with_answer=True):
    """Chat used for training (with_answer=True) and for inference (with_answer=False)."""
    msgs = [{"role": "user",
             "content": INSTRUCTION[LANG] + "\n" + format_input(row, LANG, USE_LEMMA)}]
    if with_answer:
        msgs.append({"role": "assistant", "content": row["hypersense"]})
    return msgs

def to_text(row):
    return tokenizer.apply_chat_template(make_messages(row), tokenize=False)

train_ds = Dataset.from_dict({"text": [to_text(r) for _, r in train.iterrows()]})
dev_ds   = Dataset.from_dict({"text": [to_text(r) for _, r in dev.iterrows()]})
print(train_ds[0]["text"])

# No example should be truncated
lengths = [len(tokenizer(t, add_special_tokens=False)["input_ids"]) for t in train_ds["text"]]
print(f"longest: {max(lengths)} tokens | over MAX_SEQ_LEN: {sum(l > MAX_SEQ_LEN for l in lengths)}")

## Trainer

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

steps_per_epoch = math.ceil(len(train_ds) / 16)    # effective batch size = 8 x 2
eval_every = max(steps_per_epoch // 4, 5)           # about 4 evaluations per epoch

args = SFTConfig(
    output_dir=RUN_DIR,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,        # named max_length in recent TRL versions
    packing=False,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=1 if SMOKE_TEST else 2,
    learning_rate=2e-4,
    lr_scheduler_type="linear",
    warmup_ratio=0.05,
    optim="adamw_8bit",
    weight_decay=0.01,
    fp16=not torch.cuda.is_bf16_supported(),   # T4: fp16
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=max(eval_every // 2, 1),
    eval_strategy="steps",
    eval_steps=eval_every,
    save_strategy="steps",
    save_steps=eval_every,
    save_total_limit=2,
    seed=SEED,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,               # named processing_class in recent TRL versions
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    args=args,
)

# Compute the loss on the assistant answer only (Qwen chat markers)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

In [ ]:
# Check the loss mask: positions labelled -100 are ignored, so only the answer should remain
ex = trainer.train_dataset[0]
kept = [tokenizer.pad_token_id if x == -100 else x for x in ex["labels"]]
print(repr(tokenizer.decode(kept).replace(tokenizer.pad_token, "")))

## Training

In [ ]:
t0 = time.time()
trainer.train()        # after a disconnection: trainer.train(resume_from_checkpoint=True)
print(f"training time: {(time.time() - t0) / 60:.1f} min")

In [ ]:
# Learning curves on the training set and on rand-dev
log = pd.DataFrame(trainer.state.log_history)
tr, ev = log.dropna(subset=["loss"]), log.dropna(subset=["eval_loss"])
plt.plot(tr["step"], tr["loss"], label="train")
plt.plot(ev["step"], ev["eval_loss"], label="rand-dev")
plt.xlabel("step"); plt.ylabel("loss"); plt.title(RUN_NAME); plt.legend()
plt.savefig(f"{RUN_DIR}_loss.png", dpi=150)
plt.show()

In [ ]:
# Saves the LoRA adapter only (a few dozen MB), not the base model
model.save_pretrained(f"{RUN_DIR}_adapter")
tokenizer.save_pretrained(f"{RUN_DIR}_adapter")

## Evaluation on rand-dev
Same decoding, parsing and metrics as the ICL part.

In [ ]:
FastLanguageModel.for_inference(model)

raw = generate_batch(model, tokenizer,
                     [make_messages(r, with_answer=False) for _, r in dev.iterrows()],
                     batch_size=32)
metrics, res = evaluate(dev, raw, labels)
save_results(RUN_NAME + ("_smoke" if SMOKE_TEST else ""), "rand-dev", metrics, res, OUT_DIR,
             {"method": "LoRA", "lang": LANG, "lemma": USE_LEMMA})